In [0]:
import requests
import pandas as pd

# CMS Hospital General Information
# Dataset ID: xubh-q36u
base_url = "https://data.cms.gov/provider-data/api/1/datastore/query/xubh-q36u/0"

headers = {"Accept-Encoding": "identity"}
params = {
    "limit": 500,
    "offset": 0,
    "count": "true"
}

print("Connecting to CMS Hospital General Information API...")
response = requests.get(base_url, params=params, headers=headers)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"Total records available: {data.get('count', 'unknown')}")
    df_bronze = pd.DataFrame(data['results'])
    print(f"Records retrieved: {len(df_bronze)}")
    print(f"Columns: {df_bronze.columns.tolist()}")
    display(df_bronze.head(3))
else:
    print(f"Error: {response.text}")

Connecting to CMS Hospital General Information API...
Status: 200
Total records available: 5432
Records retrieved: 500
Columns: ['facility_id', 'facility_name', 'address', 'citytown', 'state', 'zip_code', 'countyparish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footno

facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,hospital_type,hospital_ownership,emergency_services,meets_criteria_for_birthing_friendly_designation,hospital_overall_rating,hospital_overall_rating_footnote,mort_group_measure_count,count_of_facility_mort_measures,count_of_mort_measures_better,count_of_mort_measures_no_different,count_of_mort_measures_worse,mort_group_footnote,safety_group_measure_count,count_of_facility_safety_measures,count_of_safety_measures_better,count_of_safety_measures_no_different,count_of_safety_measures_worse,safety_group_footnote,readm_group_measure_count,count_of_facility_readm_measures,count_of_readm_measures_better,count_of_readm_measures_no_different,count_of_readm_measures_worse,readm_group_footnote,pt_exp_group_measure_count,count_of_facility_pt_exp_measures,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote
010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,4,,8,8,0,8,0,,8,7,3,4,0,,11,11,1,9,1,,15,15,,10,10,
010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,Yes,Y,3,,8,6,0,5,1,,8,7,0,7,0,,11,9,1,8,0,,15,15,,10,10,
010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,Yes,Y,2,,8,8,0,5,3,,8,8,4,4,0,,11,9,1,8,0,,15,15,,10,9,


In [0]:
# Write Bronze data to Delta table
spark_df = spark.createDataFrame(df_bronze)

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_hospital_general")

print(f"Bronze Delta table created: bronze_hospital_general")
print(f"Records: {df_bronze.shape[0]}")
print(f"Columns: {df_bronze.shape[1]}")

Bronze Delta table created: bronze_hospital_general
Records: 500
Columns: 38


In [0]:
# Read Bronze Delta table
df_bronze = spark.table("bronze_hospital_general").toPandas()

# Select and rename key columns
silver_cols = [
    'facility_id',
    'facility_name',
    'citytown',
    'state',
    'zip_code',
    'countyparish',
    'hospital_type',
    'hospital_ownership',
    'emergency_services',
    'hospital_overall_rating',
    'count_of_safety_measures_better',
    'count_of_safety_measures_worse',
    'count_of_readm_measures_better',
    'count_of_readm_measures_worse',
    'count_of_mort_measures_better',
    'count_of_mort_measures_worse'
]

df_silver = df_bronze[silver_cols].copy()

# Rename for readability
df_silver.columns = [
    'facility_id',
    'facility_name',
    'city',
    'state',
    'zip_code',
    'county',
    'hospital_type',
    'ownership',
    'emergency_services',
    'overall_rating',
    'safety_better',
    'safety_worse',
    'readmission_better',
    'readmission_worse',
    'mortality_better',
    'mortality_worse'
]

# Fix data types
import pandas as pd
df_silver['overall_rating'] = pd.to_numeric(df_silver['overall_rating'], errors='coerce')
df_silver['safety_better'] = pd.to_numeric(df_silver['safety_better'], errors='coerce')
df_silver['safety_worse'] = pd.to_numeric(df_silver['safety_worse'], errors='coerce')
df_silver['readmission_better'] = pd.to_numeric(df_silver['readmission_better'], errors='coerce')
df_silver['readmission_worse'] = pd.to_numeric(df_silver['readmission_worse'], errors='coerce')
df_silver['mortality_better'] = pd.to_numeric(df_silver['mortality_better'], errors='coerce')
df_silver['mortality_worse'] = pd.to_numeric(df_silver['mortality_worse'], errors='coerce')

# Data quality flag
df_silver['data_quality_flag'] = df_silver['overall_rating'].isna()

print(f"Silver records: {len(df_silver)}")
print(f"Columns: {df_silver.shape[1]}")
print(f"Hospitals with overall rating: {df_silver['overall_rating'].notna().sum()}")
print(f"Data quality flags: {df_silver['data_quality_flag'].sum()}")
display(df_silver.head(3))

Silver records: 500
Columns: 17
Hospitals with overall rating: 335
Data quality flags: 165


facility_id,facility_name,city,state,zip_code,county,hospital_type,ownership,emergency_services,overall_rating,safety_better,safety_worse,readmission_better,readmission_worse,mortality_better,mortality_worse,data_quality_flag
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,HOUSTON,Acute Care Hospitals,Government - Hospital District or Authority,Yes,4.0,3.0,0.0,1.0,1.0,0.0,0.0,false
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,MARSHALL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,3.0,0.0,0.0,1.0,0.0,0.0,1.0,false
010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,LAUDERDALE,Acute Care Hospitals,Proprietary,Yes,2.0,4.0,0.0,1.0,0.0,0.0,3.0,false


In [0]:
# Write Silver to Delta table
silver_spark = spark.createDataFrame(df_silver)

silver_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_hospital_general")

print("Silver Delta table created: silver_hospital_general")
print(f"Records: {len(df_silver)}")
print(f"Rated hospitals: {df_silver['overall_rating'].notna().sum()}")
print(f"Flagged records: {df_silver['data_quality_flag'].sum()}")

Silver Delta table created: silver_hospital_general
Records: 500
Rated hospitals: 335
Flagged records: 165


In [0]:
import pandas as pd

# Read Silver Delta table
df_silver = spark.table("silver_hospital_general").toPandas()

# GOLD TABLE 1 - Average rating by state
gold_by_state = df_silver[df_silver['overall_rating'].notna()].groupby('state').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count'),
    rated_count=('overall_rating', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 2 - Average rating by hospital type
gold_by_type = df_silver[df_silver['overall_rating'].notna()].groupby('hospital_type').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 3 - Average rating by ownership
gold_by_ownership = df_silver[df_silver['overall_rating'].notna()].groupby('ownership').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 4 - Top 20 rated hospitals
gold_top_hospitals = df_silver[df_silver['overall_rating'] == 5][
    ['facility_id', 'facility_name', 'city', 'state', 'hospital_type', 'ownership', 'overall_rating']
].head(20)

print(f"Gold by state: {len(gold_by_state)} rows")
print(f"Gold by type: {len(gold_by_type)} rows")
print(f"Gold by ownership: {len(gold_by_ownership)} rows")
print(f"Top rated hospitals: {len(gold_top_hospitals)} rows")

print("\nTop 5 States by Average Rating:")
display(gold_by_state.head())

print("\nRating by Hospital Type:")
display(gold_by_type)

Gold by state: 5 rows
Gold by type: 3 rows
Gold by ownership: 11 rows
Top rated hospitals: 20 rows

Top 5 States by Average Rating:


state,avg_rating,hospital_count,rated_count
CA,3.15,167,167
AZ,3.11,55,55
AK,2.9,10,10
AR,2.84,44,44
AL,2.71,59,59



Rating by Hospital Type:


hospital_type,avg_rating,hospital_count
Acute Care - Veterans Administration,4.36,11
Critical Access Hospitals,3.42,12
Acute Care Hospitals,2.96,312


In [0]:
# Write all Gold tables to Delta
spark.createDataFrame(gold_by_state).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_state")

spark.createDataFrame(gold_by_type).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_type")

spark.createDataFrame(gold_by_ownership).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_ownership")

spark.createDataFrame(gold_top_hospitals).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_top_rated_hospitals")

print("All Gold Delta tables created successfully!")
print("  gold_hospital_by_state")
print("  gold_hospital_by_type")
print("  gold_hospital_by_ownership")
print("  gold_top_rated_hospitals")

All Gold Delta tables created successfully!
  gold_hospital_by_state
  gold_hospital_by_type
  gold_hospital_by_ownership
  gold_top_rated_hospitals
